# Create Training Labels from the Critical Mineral Deposits Database
from shapefile to GeoTIFF


In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd

from beak.utilities.io import  data_folder, load_raster
from beak.utilities.conversions import create_binary_raster


# Load data

**User definitions**

In [2]:
BASE_PATH = data_folder()
EPSG_CODE = 102008
RESOLUTION = 500

BASE_EXTENT = "tusk_great_basin"
BASE_RASTER = BASE_PATH / "processed" / str(f"regional_{BASE_EXTENT}_{EPSG_CODE}_{RESOLUTION}") / "base_raster" / "template_raster.tif"
CMA_EXTENT = "regional"

# Export
# If some data sets are already **LOG** scaled they need special actions. They will be unified and stored in a separate folder.
PATH_SHAPEFILE = BASE_PATH / "raw" / "mineral_deposits" / "Tungsten_Skarn" / "regional_tusk_great_basin" / "h4_tungsten_sites_0822a.gpkg"
OUT_FILE = BASE_PATH / "processed" / str(f"{CMA_EXTENT}_{BASE_EXTENT}_{EPSG_CODE}_{RESOLUTION}") / "labels" / "TA2_USGS_0822a.tif"

print(f"Input: {PATH_SHAPEFILE}")
print(f"Export: {OUT_FILE}")

Input: S:\Projekte\20230082_DARPA_CriticalMAAS_TA3\Bearbeitung\GitHub\beak-ta3\src\beak\data\raw\mineral_deposits\Tungsten_Skarn\regional_tusk_great_basin\h4_tungsten_sites_0822a.gpkg
Export: S:\Projekte\20230082_DARPA_CriticalMAAS_TA3\Bearbeitung\GitHub\beak-ta3\src\beak\data\processed\regional_tusk_great_basin_102008_500\labels\TA2_USGS_0822a.tif


# Create Labels


Reproject to **BASE RASTER** CRS

In [3]:
from beak.utilities.vector_processing import _reproject_vector_data

base_raster = load_raster(BASE_RASTER)
template_crs = base_raster.crs

gdf_ta2 = gpd.read_file(PATH_SHAPEFILE, layer="h4_tungsten_sites_merged_dedup_v2_outside_usgs_roi")
gdf_ta2 = _reproject_vector_data(gdf_ta2, crs=template_crs, encoding="utf-8", query=None)

gdf_usgs = gpd.read_file(PATH_SHAPEFILE, layer="usgs_sites_in_roi")
gdf_usgs = _reproject_vector_data(gdf_usgs, crs=template_crs, encoding="utf-8", query=None)

gdf = pd.concat([gdf_ta2.geometry, gdf_usgs.geometry])


Create binary label raster

In [4]:
out_array = create_binary_raster(gdf, base_raster, query=None, all_touched=False, same_shape=True, fill_negatives=True, out_file=OUT_FILE)
print(f"Number of positive training labels: {np.sum(out_array==1)}")

Number of positive training labels: 856
